
Information about tables are down below. [](url)

In [0]:
SOURCE = "bigquery_spotify_catalog.spotify_dataset.spotify_cleaned"

def build(name: str, sql: str):
    (spark.sql(sql).write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"workspace.spotify.{name}"))
    print(f"built {name}")

In [0]:
build("headline_stats", f"""
    SELECT
      ROUND(SUM(seconds_played)/3600, 1) AS total_hours,
      COUNT(DISTINCT artist) AS artists,
      COUNT(DISTINCT spotify_track_uri) AS tracks,
      ROUND(AVG(CASE WHEN skipped THEN 1.0 ELSE 0 END), 3) AS skip_rate
    FROM {SOURCE}
""")
 
build("top_artists", f"""
    SELECT artist, SUM(seconds_played)/3600 AS hours, COUNT(*) AS plays
    FROM {SOURCE}
    GROUP BY artist
    ORDER BY hours DESC
    LIMIT 50
""")
 
build("top_tracks", f"""
    SELECT song_name, artist, spotify_track_uri, COUNT(*) AS plays
    FROM {SOURCE}
    GROUP BY song_name, artist, spotify_track_uri
    ORDER BY plays DESC
    LIMIT 50
""")
 
build("daily_listening", f"""
    SELECT DATE(ts) AS play_date, SUM(seconds_played)/3600 AS hours
    FROM {SOURCE}
    GROUP BY play_date
    ORDER BY play_date
""")
 
build("plays_by_hour", f"""
    SELECT HOUR(ts) AS hour_of_day, COUNT(*) AS plays
    FROM {SOURCE}
    GROUP BY hour_of_day
    ORDER BY hour_of_day
""")
 
 
# -------------------------------------------------------------
# Layer 2 — sessionization (gaps and islands)
#
# A new session starts when more than 30 minutes pass with no
# plays. Each play is also labelled by outcome: anything under
# 10 seconds was skipped past without being heard.
# -------------------------------------------------------------
 
build("session_plays", f"""
    WITH gaps AS (
      SELECT *,
             TIMESTAMPDIFF(MINUTE,
               LAG(ts) OVER (ORDER BY ts), ts) AS mins_since_last
      FROM {SOURCE}
    )
    SELECT *,
           SUM(CASE WHEN mins_since_last > 30 OR mins_since_last IS NULL
                    THEN 1 ELSE 0 END) OVER (ORDER BY ts) AS session_id,
           seconds_played >= 10 AS is_real_play,
           CASE WHEN seconds_played < 10 THEN 'skipped through'
                WHEN reason_end = 'trackdone' THEN 'played fully'
                ELSE 'partial listen' END AS play_outcome
    FROM gaps
""")
 
 
# -------------------------------------------------------------
# Layer 3 — one row per session
#
# Two independent measures:
#   intent_ratio        share of real plays that were hand-picked
#   skip_through_ratio  share of plays skipped past under 10s
#
# Thresholds come from the observed distribution rather than
# round numbers:
#   intent_ratio        p75 = 0.14   p90 = 0.35
#   skip_through_ratio  p75 = 0.32
# -------------------------------------------------------------
 
build("sessions", """
    SELECT session_id,
           MIN(ts) AS started_at,
           COUNT(*) AS plays,
           COUNT_IF(seconds_played >= 10) AS real_plays,
           ROUND(SUM(seconds_played)/60, 1) AS minutes,
           COUNT(DISTINCT artist) AS artists,
           ROUND(COUNT_IF(seconds_played < 10) / COUNT(*), 2) AS skip_through_ratio,
           ROUND(
             COUNT_IF(seconds_played >= 10 AND reason_start IN ('clickrow','playbtn'))
             / NULLIF(COUNT_IF(seconds_played >= 10), 0), 2) AS intent_ratio,
           CASE
             WHEN COUNT_IF(seconds_played >= 10 AND reason_start IN ('clickrow','playbtn'))
                  / NULLIF(COUNT_IF(seconds_played >= 10), 0) >= 0.35 THEN 'active'
             WHEN COUNT_IF(seconds_played >= 10 AND reason_start IN ('clickrow','playbtn'))
                  / NULLIF(COUNT_IF(seconds_played >= 10), 0) >= 0.14 THEN 'mixed'
             ELSE 'passive'
           END AS session_type,
           CASE
             WHEN COUNT_IF(seconds_played < 10) / COUNT(*) >= 0.32 THEN 'restless'
             ELSE 'settled'
           END AS listening_mode
    FROM workspace.spotify.session_plays
    GROUP BY session_id
    ORDER BY started_at
""")
 
 
# -------------------------------------------------------------
# Layer 4 — session rollups for the dashboard
# -------------------------------------------------------------
 
build("session_type_summary", """
    SELECT session_type,
           COUNT(*) AS sessions,
           ROUND(AVG(minutes), 1) AS avg_minutes,
           ROUND(SUM(minutes)/60, 1) AS total_hours,
           ROUND(AVG(artists), 1) AS avg_artists,
           ROUND(AVG(skip_through_ratio), 2) AS avg_skip_through
    FROM workspace.spotify.sessions
    GROUP BY session_type
""")
 
build("session_type_by_month", """
    SELECT DATE_TRUNC('MONTH', started_at) AS month,
           session_type,
           COUNT(*) AS sessions
    FROM workspace.spotify.sessions
    GROUP BY month, session_type
    ORDER BY month, session_type
""")
 
build("session_quadrants", """
    SELECT session_type, listening_mode,
           COUNT(*) AS sessions,
           ROUND(AVG(minutes), 1) AS avg_minutes
    FROM workspace.spotify.sessions
    GROUP BY session_type, listening_mode
    ORDER BY sessions DESC
""")
 


built headline_stats
built top_artists
built top_tracks
built daily_listening
built plays_by_hour
built session_plays
built sessions
built session_type_summary
built session_type_by_month
built session_quadrants


## **What these tables are and why**

**Layer 1 — direct aggregates**

> `headline_stats`, `top_artists`, `top_tracks`, `daily_listening`,
> `plays_by_hour`

Straight rollups of the play history — totals, rankings, time series.
These answer *what* was listened to.

**Layer 2 — sessionization**

> `session_plays` — every play, tagged with the session it belongs to

Raw history is a flat list of timestamps. The interesting questions are
about sessions: stretches of continuous listening separated by breaks.
This table tags every play with a session id using the gaps-and-islands
pattern — `LAG(ts)` gives the gap before each play, a flag fires when
that gap exceeds 30 minutes, and a running `SUM` over those flags becomes
the session id, incrementing only at a break. Each play is also labelled
by how much was actually heard, since anything under 10 seconds was
skipped past rather than listened to.

**Layer 3 — session summary**

> `sessions` — one row per session
>
> `intent_ratio` — share of heard tracks deliberately picked
> (`clickrow` or `playbtn`) rather than autoplayed
>
> `skip_through_ratio` — share of tracks skipped past unheard

Two measures that describe *how* listening happened rather than what was
played. They're separate axes, not inverses — a session can be low on
both (an album left running) or high on both (hunting for something
specific, picking and rejecting).

**Layer 4 — rollups**

> `session_type_summary`, `session_type_by_month`, `session_quadrants`

Small tables shaped for specific dashboard components, so the frontend
reads pre-computed results instead of aggregating at render time.

---

## **Why are plays under 10 seconds excluded from intent?**

An earlier version counted every play toward `intent_ratio`, including
tracks skipped after one second, and produced a result that was clearly
wrong.

Session 6529 opens with five tracks lasting 6, 5, 1, 1 and 3 seconds,
then settles at 08:13 into a track played in full. That is someone
actively hunting for something to listen to — about as engaged as
listening gets — and the model labelled it passive, because the rapid
skips diluted the ratio.

Excluding plays under 10 seconds means intent is judged only on tracks
that were actually heard. The skipping behaviour isn't discarded; it
moves to `skip_through_ratio`, where it belongs.

---

## **Where did the thresholds come from?**

The `session_type` and `listening_mode` cutoffs aren't round numbers —
they're percentiles of the observed distributions.

The first version split active from passive at `0.4`, picked
arbitrarily. Nearly every session came back passive, which raised the
question of whether that reflected real behaviour or just a badly placed
line.

*Describing the distribution answered it:*

```sql
SELECT
  ROUND(AVG(intent_ratio), 3) AS mean,
  ROUND(PERCENTILE(intent_ratio, 0.10), 3) AS p10,
  ROUND(PERCENTILE(intent_ratio, 0.25), 3) AS p25,
  ROUND(PERCENTILE(intent_ratio, 0.50), 3) AS median,
  ROUND(PERCENTILE(intent_ratio, 0.75), 3) AS p75,
  ROUND(PERCENTILE(intent_ratio, 0.90), 3) AS p90,
  ROUND(MIN(intent_ratio), 3) AS min,
  ROUND(MAX(intent_ratio), 3) AS max
FROM workspace.spotify.sessions;
```

> **Result** — mean `0.122` | p10 `0` | p25 `0` | median `0.03` |
> p75 `0.14` | p90 `0.35` | min `0` | max `1`

Heavily right-skewed. At least a quarter of sessions contain no
hand-picked plays at all — p25 sits at zero. The median session is
`0.03`, which in practice means one deliberate pick followed by autoplay
for the rest. The original `0.4` sat **above p90**, so fewer than one
session in ten could ever have qualified as active. It was measuring the
threshold, not the behaviour.

**Why percentiles rather than the mean?**

Note the mean (`0.122`) sitting four times above the median (`0.03`) — a
handful of sessions near 1.0 drag it upward. On a distribution this
skewed the mean describes a session that doesn't really exist. A
percentile makes a verifiable claim instead: p75 means exactly
three-quarters of sessions fall beneath it.

> Adopted — **p90 (0.35)** for active, **p75 (0.14)** for mixed

*Same approach for the second axis:*

```sql
SELECT
  ROUND(AVG(skip_through_ratio), 3) AS mean,
  ROUND(PERCENTILE(skip_through_ratio, 0.10), 3) AS p10,
  ROUND(PERCENTILE(skip_through_ratio, 0.25), 3) AS p25,
  ROUND(PERCENTILE(skip_through_ratio, 0.50), 3) AS median,
  ROUND(PERCENTILE(skip_through_ratio, 0.75), 3) AS p75,
  ROUND(PERCENTILE(skip_through_ratio, 0.90), 3) AS p90
FROM workspace.spotify.sessions;
```

> **Result** — mean `0.192` | p10 `0` | p25 `0` | median `0.06` |
> p75 `0.32` | p90 `0.61`

The same skew with a heavier tail. A typical session contains almost no
skip-throughs and at least a quarter contain none at all — but the top
ten percent are *more than half* skip-throughs, sessions spent searching
rather than listening.

> Adopted — **p75 (0.32)** as the line between restless and settled

Both queries are worth re-running whenever a ratio definition changes,
since the thresholds are derived from their output and drift out of date
otherwise.

---

## **How do you inspect a single session?**

> `sessions` holds one row per session, `session_plays` holds every
> individual play, and `session_id` joins them

*The join:*

```sql
SELECT sesh.session_id, sesh.started_at, sesh.session_type,
       sesh.listening_mode, sesh_play.ts, sesh_play.song_name,
       sesh_play.artist, sesh_play.seconds_played, sesh_play.play_outcome
FROM workspace.spotify.sessions AS sesh
INNER JOIN workspace.spotify.session_plays AS sesh_play
  ON sesh.session_id = sesh_play.session_id
WHERE sesh.started_at >= '2022-01-01'
  AND sesh.started_at <  '2022-01-02'
ORDER BY sesh_play.ts;
```

Columns are listed explicitly rather than using `SELECT *`. Both tables
contain a `session_id` column, and `SELECT *` returns both, making the
output ambiguous — an earlier version of this query silently produced a
cross join, caught only by noticing that one side held the same value on
every row.

The date filter uses `>= start AND < next_day` rather than `BETWEEN`,
which is inclusive at both ends and would pull in anything falling
exactly on midnight of the following day.
